In [13]:
import pandas as pd
import numpy as np

file_path_202604 = "38个浮游植物样品数据-2026.04样品.xlsx"

df_202604 = pd.read_excel(
    file_path_202604,
    sheet_name="原始数据"
)

df_202604.columns = (
    df_202604.columns
    .astype(str)
    .str.strip()
)

print(df_202604.columns.tolist())
print(df_202604.head())

['样品编号', '采样体积（L）', '浓缩体积（mL）', '计数体积（mL）', '中文名', '拉丁名', '类群', '个体数量']
      样品编号  采样体积（L）  浓缩体积（mL）  计数体积（mL）    中文名                        拉丁名  类群  \
0  XS-11 B        2        40         1   卵形藻属              Cocconeis sp.  硅藻   
1  XS-11 B        2        40         1  串珠梯楔藻  Climacosphenia moniligera  硅藻   
2  XS-11 B        2        40         1    羽纹藻             Pinnularia sp.  硅藻   
3  XS-11 B        2        40         1   小环藻属             Cyclotella sp.  硅藻   
4  XS-11 B        2        40         1   长菱形藻       Nitzschia longissima  硅藻   

   个体数量  
0     8  
1     2  
2    13  
3     9  
4    14  


In [14]:
text_columns = [
    "样品编号",
    "中文名",
    "拉丁名",
    "类群"
]

for col in text_columns:
    df_202604[col] = (
        df_202604[col]
        .astype("string")
        .str.strip()
    )

In [15]:
numeric_columns = [
    "采样体积（L）",
    "浓缩体积（mL）",
    "计数体积（mL）",
    "个体数量"
]

for col in numeric_columns:
    df_202604[col] = pd.to_numeric(
        df_202604[col],
        errors="coerce"
    )

### 计算每行物种丰度

In [16]:
df_202604["丰度_cells_L"] = (
    df_202604["个体数量"]
    * df_202604["浓缩体积（mL）"]
    / (
        df_202604["计数体积（mL）"]
        * df_202604["采样体积（L）"]
    )
)

In [17]:
print(
    df_202604[
        [
            "样品编号",
            "中文名",
            "拉丁名",
            "类群",
            "个体数量",
            "丰度_cells_L"
        ]
    ].head(20)
)

       样品编号       中文名                        拉丁名  类群  个体数量  丰度_cells_L
0   XS-11 B      卵形藻属              Cocconeis sp.  硅藻     8       160.0
1   XS-11 B     串珠梯楔藻  Climacosphenia moniligera  硅藻     2        40.0
2   XS-11 B       羽纹藻             Pinnularia sp.  硅藻    13       260.0
3   XS-11 B      小环藻属             Cyclotella sp.  硅藻     9       180.0
4   XS-11 B      长菱形藻       Nitzschia longissima  硅藻    14       280.0
5   XS-11 B  亚德里亚海杆线藻      Rhabdonema adriaticum  硅藻     2        40.0
6   XS-11 B      胸隔藻属             Mastogloia sp.  硅藻     3        60.0
7   XS-11 B      圆筛藻属          Coscinodiscus sp.  硅藻    21       420.0
8   XS-11 B     铁氏束毛藻   Trichodesmium thiebautii  蓝藻     1        20.0
9   XS-11 B     短纹楔形藻      Licmophora abbreviata  硅藻     3        60.0
10  XS-11 B      脆杆藻属             Fragilaria sp.  硅藻    12       240.0
11  XS-11 B      环沟藻属             Gyrodinium sp.  甲藻     4        80.0
12  XS-11 B      角毛藻属            Chaetoceros sp.  硅藻     6       120.0
13  XS

### 合并同一样品中的重复物种

In [18]:
species_sample_202604 = (
    df_202604.groupby(
        [
            "样品编号",
            "中文名",
            "拉丁名",
            "类群"
        ],
        as_index=False
    )
    .agg(
        个体数量=("个体数量", "sum"),
        丰度_cells_L=("丰度_cells_L", "sum")
    )
)

## 一、全部样品分析

In [19]:
total_samples_202604 = (
    species_sample_202604["样品编号"].nunique()
)

print("2026.04 全部样品数：", total_samples_202604)

2026.04 全部样品数： 37


In [20]:
# 1. 全部样品三大类群丰度
target_groups = ["硅藻", "甲藻", "蓝藻"]

three_groups_202604 = species_sample_202604[
    species_sample_202604["类群"].isin(target_groups)
].copy()

group_abundance_202604 = (
    three_groups_202604
    .groupby(
        ["样品编号", "类群"],
        as_index=False
    )["丰度_cells_L"]
    .sum()
    .pivot(
        index="样品编号",
        columns="类群",
        values="丰度_cells_L"
    )
    .fillna(0)
    .reset_index()
)

for group in target_groups:
    if group not in group_abundance_202604.columns:
        group_abundance_202604[group] = 0

group_abundance_202604 = (
    group_abundance_202604[
        ["样品编号", "硅藻", "甲藻", "蓝藻"]
    ]
    .rename(
        columns={
            "硅藻": "硅藻丰度_cells_L",
            "甲藻": "甲藻丰度_cells_L",
            "蓝藻": "蓝藻丰度_cells_L"
        }
    )
)

In [21]:
# 2. 全部样品总丰度
total_abundance_202604 = (
    species_sample_202604
    .groupby(
        "样品编号",
        as_index=False
    )["丰度_cells_L"]
    .sum()
    .rename(
        columns={
            "丰度_cells_L":
            "浮游植物总丰度_cells_L"
        }
    )
)
# 合并：
sample_summary_202604 = (
    group_abundance_202604
    .merge(
        total_abundance_202604,
        on="样品编号",
        how="outer"
    )
    .fillna(0)
)
# 其他类群丰度：
sample_summary_202604[
    "其他类群丰度_cells_L"
] = (
    sample_summary_202604[
        "浮游植物总丰度_cells_L"
    ]
    - sample_summary_202604[
        "硅藻丰度_cells_L"
    ]
    - sample_summary_202604[
        "甲藻丰度_cells_L"
    ]
    - sample_summary_202604[
        "蓝藻丰度_cells_L"
    ]
)

In [22]:
# 3. 全部样品三大类群相对丰度
for group in ["硅藻", "甲藻", "蓝藻"]:
    sample_summary_202604[
        f"{group}相对丰度"
    ] = (
        sample_summary_202604[
            f"{group}丰度_cells_L"
        ]
        / sample_summary_202604[
            "浮游植物总丰度_cells_L"
        ]
    )
# 处理异常值：
relative_columns_202604 = [
    "硅藻相对丰度",
    "甲藻相对丰度",
    "蓝藻相对丰度"
]

sample_summary_202604[
    relative_columns_202604
] = (
    sample_summary_202604[
        relative_columns_202604
    ]
    .replace([np.inf, -np.inf], np.nan)
    .fillna(0)
)

##  二、全部样品优势种

In [23]:
# 1. 按物种汇总
species_summary_202604 = (
    species_sample_202604
    .groupby(
        ["中文名", "拉丁名", "类群"],
        as_index=False
    )
    .agg(
        物种总丰度_n_i=(
            "丰度_cells_L",
            "sum"
        ),
        出现样品数=(
            "样品编号",
            "nunique"
        )
    )
)

In [24]:
# 2. 计算全部累计丰度 N
N_202604 = (
    species_summary_202604[
        "物种总丰度_n_i"
    ].sum()
)

In [25]:
# 3. 计算出现频率、相对丰度和优势度
species_summary_202604[
    "出现频率_f_i"
] = (
    species_summary_202604[
        "出现样品数"
    ]
    / total_samples_202604
)

species_summary_202604[
    "相对丰度_n_i_N"
] = (
    species_summary_202604[
        "物种总丰度_n_i"
    ]
    / N_202604
)

species_summary_202604[
    "优势度_Y"
] = (
    species_summary_202604[
        "相对丰度_n_i_N"
    ]
    * species_summary_202604[
        "出现频率_f_i"
    ]
)

species_summary_202604[
    "是否优势种"
] = np.where(
    species_summary_202604[
        "优势度_Y"
    ] > 0.02,
    "优势种",
    "非优势种"
)

In [26]:
# 排序并提取优势种：
species_summary_202604 = (
    species_summary_202604
    .sort_values(
        "优势度_Y",
        ascending=False
    )
    .reset_index(drop=True)
)

dominant_species_202604 = (
    species_summary_202604[
        species_summary_202604[
            "优势度_Y"
        ] > 0.02
    ]
    .copy()
)

## 三、再分析 B 层

In [27]:
# 从全部样品基础表中筛选：
b_data_202604 = species_sample_202604[
    species_sample_202604[
        "样品编号"
    ].str.endswith(" B", na=False)
].copy()

In [28]:
# 统计 B 层样品数：
b_total_samples_202604 = (
    b_data_202604["样品编号"].nunique()
)

print(
    "2026.04 B层样品数：",
    b_total_samples_202604
)

2026.04 B层样品数： 16


## 四、B 层三大类群丰度和相对丰度

In [29]:
b_three_groups_202604 = b_data_202604[
    b_data_202604["类群"].isin(target_groups)
].copy()

b_group_abundance_202604 = (
    b_three_groups_202604
    .groupby(
        ["样品编号", "类群"],
        as_index=False
    )["丰度_cells_L"]
    .sum()
    .pivot(
        index="样品编号",
        columns="类群",
        values="丰度_cells_L"
    )
    .fillna(0)
    .reset_index()
)

for group in target_groups:
    if group not in b_group_abundance_202604.columns:
        b_group_abundance_202604[group] = 0

b_group_abundance_202604 = (
    b_group_abundance_202604[
        ["样品编号", "硅藻", "甲藻", "蓝藻"]
    ]
    .rename(
        columns={
            "硅藻": "硅藻丰度_cells_L",
            "甲藻": "甲藻丰度_cells_L",
            "蓝藻": "蓝藻丰度_cells_L"
        }
    )
)
# B 层总丰度：
b_total_abundance_202604 = (
    b_data_202604
    .groupby(
        "样品编号",
        as_index=False
    )["丰度_cells_L"]
    .sum()
    .rename(
        columns={
            "丰度_cells_L":
            "浮游植物总丰度_cells_L"
        }
    )
)
# 合并：
b_sample_summary_202604 = (
    b_group_abundance_202604
    .merge(
        b_total_abundance_202604,
        on="样品编号",
        how="outer"
    )
    .fillna(0)
)
# 其他类群：
b_sample_summary_202604[
    "其他类群丰度_cells_L"
] = (
    b_sample_summary_202604[
        "浮游植物总丰度_cells_L"
    ]
    - b_sample_summary_202604[
        "硅藻丰度_cells_L"
    ]
    - b_sample_summary_202604[
        "甲藻丰度_cells_L"
    ]
    - b_sample_summary_202604[
        "蓝藻丰度_cells_L"
    ]
)
# B 层相对丰度：
for group in ["硅藻", "甲藻", "蓝藻"]:
    b_sample_summary_202604[
        f"{group}相对丰度"
    ] = (
        b_sample_summary_202604[
            f"{group}丰度_cells_L"
        ]
        / b_sample_summary_202604[
            "浮游植物总丰度_cells_L"
        ]
    )

## 五、B 层优势种

In [30]:
b_species_summary_202604 = (
    b_data_202604
    .groupby(
        ["中文名", "拉丁名", "类群"],
        as_index=False
    )
    .agg(
        物种总丰度_n_i=(
            "丰度_cells_L",
            "sum"
        ),
        出现样品数=(
            "样品编号",
            "nunique"
        )
    )
)

In [31]:
# 计算 N：
b_N_202604 = (
    b_species_summary_202604[
        "物种总丰度_n_i"
    ].sum()
)
# 计算频率、相对丰度和优势度：
b_species_summary_202604[
    "出现频率_f_i"
] = (
    b_species_summary_202604[
        "出现样品数"
    ]
    / b_total_samples_202604
)

b_species_summary_202604[
    "相对丰度_n_i_N"
] = (
    b_species_summary_202604[
        "物种总丰度_n_i"
    ]
    / b_N_202604
)

b_species_summary_202604[
    "优势度_Y"
] = (
    b_species_summary_202604[
        "相对丰度_n_i_N"
    ]
    * b_species_summary_202604[
        "出现频率_f_i"
    ]
)

b_species_summary_202604[
    "是否优势种"
] = np.where(
    b_species_summary_202604[
        "优势度_Y"
    ] > 0.02,
    "优势种",
    "非优势种"
)
# 排序和提取：
b_species_summary_202604 = (
    b_species_summary_202604
    .sort_values(
        "优势度_Y",
        ascending=False
    )
    .reset_index(drop=True)
)

b_dominant_species_202604 = (
    b_species_summary_202604[
        b_species_summary_202604[
            "优势度_Y"
        ] > 0.02
    ]
    .copy()
)

## 六、检查两套结果

In [32]:
print("===== 2026.04 全部样品 =====")
print("样品数：", total_samples_202604)
print("累计总丰度：", N_202604)
print(
    "相对丰度之和：",
    species_summary_202604[
        "相对丰度_n_i_N"
    ].sum()
)
print(dominant_species_202604)

print("\n===== 2026.04 B层 =====")
print("B层样品数：", b_total_samples_202604)
print("B层累计总丰度：", b_N_202604)
print(
    "相对丰度之和：",
    b_species_summary_202604[
        "相对丰度_n_i_N"
    ].sum()
)
print(b_dominant_species_202604)

===== 2026.04 全部样品 =====
样品数： 37
累计总丰度： 90300.0
相对丰度之和： 0.9999999999999999
     中文名                   拉丁名  类群  物种总丰度_n_i  出现样品数  出现频率_f_i  相对丰度_n_i_N  \
0   裸甲藻属       Gymnodinium sp.  甲藻    24200.0     32  0.864865    0.267996   
1   菱形藻属         Nitzschia sp.  硅藻    11000.0     37  1.000000    0.121816   
2   舟形藻属          Navicula sp.  硅藻     7420.0     35  0.945946    0.082171   
3   环沟藻属        Gyrodinium sp.  甲藻     5800.0     31  0.837838    0.064230   
4  标志星杆藻   Asterionella notata  硅藻     5760.0     26  0.702703    0.063787   
5   针杆藻属           Synedra sp.  硅藻     4280.0     33  0.891892    0.047398   
6   长菱形藻  Nitzschia longissima  硅藻     4320.0     28  0.756757    0.047841   
7    羽纹藻        Pinnularia sp.  硅藻     2900.0     31  0.837838    0.032115   

      优势度_Y 是否优势种  
0  0.231780   优势种  
1  0.121816   优势种  
2  0.077729   优势种  
3  0.053815   优势种  
4  0.044824   优势种  
5  0.042274   优势种  
6  0.036204   优势种  
7  0.026907   优势种  

===== 2026.04 B层 =====
B层样品数： 16
B层累计总丰度：

## 七、导出一个 Excel，包含全部样品和 B 层两套结果

In [33]:
output_file_202604 = (
    "2026.04_浮游植物丰度与优势种结果.xlsx"
)

all_sample_list_202604 = pd.DataFrame({
    "样品编号": sorted(
        species_sample_202604[
            "样品编号"
        ].unique()
    )
})

b_sample_list_202604 = pd.DataFrame({
    "样品编号": sorted(
        b_data_202604[
            "样品编号"
        ].unique()
    )
})

with pd.ExcelWriter(
    output_file_202604,
    engine="openpyxl"
) as writer:

    all_sample_list_202604.to_excel(
        writer,
        sheet_name="全部样品清单",
        index=False
    )

    species_sample_202604.to_excel(
        writer,
        sheet_name="全部样品物种丰度",
        index=False
    )

    sample_summary_202604.to_excel(
        writer,
        sheet_name="全部类群丰度",
        index=False
    )

    species_summary_202604.to_excel(
        writer,
        sheet_name="全部物种优势度",
        index=False
    )

    dominant_species_202604.to_excel(
        writer,
        sheet_name="全部优势种",
        index=False
    )

    b_sample_list_202604.to_excel(
        writer,
        sheet_name="B层样品清单",
        index=False
    )

    b_data_202604.to_excel(
        writer,
        sheet_name="B层样品物种丰度",
        index=False
    )

    b_sample_summary_202604.to_excel(
        writer,
        sheet_name="B层类群丰度",
        index=False
    )

    b_species_summary_202604.to_excel(
        writer,
        sheet_name="B层全部物种优势度",
        index=False
    )

    b_dominant_species_202604.to_excel(
        writer,
        sheet_name="B层优势种",
        index=False
    )

print(
    f"结果已保存到：{output_file_202604}"
)

结果已保存到：2026.04_浮游植物丰度与优势种结果.xlsx
